In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# LABELING OTOMATIS SENTIMEN
# =========================================================

# 1. LOAD DATA
file_path = "hasil_preprocessing(1).csv"
df = pd.read_csv(file_path, sep=";", encoding="utf-8-sig")

# 2. AMBIL HANYA KOLOM hasil_preprocessing
df = df[["hasil_preprocessing"]].copy()

# 3. BERSIHKAN DATA
df["hasil_preprocessing"] = df["hasil_preprocessing"].astype(str).str.strip()
df["hasil_preprocessing"] = df["hasil_preprocessing"].replace("nan", np.nan)
df = df.dropna(subset=["hasil_preprocessing"])
df = df[df["hasil_preprocessing"] != ""]

print("Jumlah data:", len(df))

# =========================================================
# 4. LOAD INSET LEXICON 10.000+
# =========================================================
url_positive = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
url_negative = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

positive_lexicon = pd.read_csv(url_positive, sep="\t")
negative_lexicon = pd.read_csv(url_negative, sep="\t")

positive_dict = dict(zip(positive_lexicon["word"].astype(str), positive_lexicon["weight"]))
negative_dict = dict(zip(negative_lexicon["word"].astype(str), abs(negative_lexicon["weight"])))

# =========================================================
# 5.KAMUS KHUSUS
# =========================================================

custom_positive = {
    "bagus": 3, "baik": 3, "mantap": 3, "keren": 3, "hebat": 3, "top": 3,
    "murah": 2, "aman": 3, "stabil": 2, "kuat": 3, "cepat": 2, "lancar": 3,
    "sukses": 4, "positif": 3, "optimis": 3, "menarik": 2, "unggul": 3,
    "potensi": 2, "prospek": 3, "menjanjikan": 4, "rekomendasi": 3,
    "profit": 4, "untung": 4, "cuan": 4, "naik": 3, "melonjak": 4,
    "tumbuh": 3, "bullish": 4, "pump": 4, "breakout": 4, "rebound": 3,
    "recovery": 3, "akumulasi": 3, "buy": 3, "buying": 3, "long": 3,
    "support kuat": 4, "sinyal bullish": 5, "tren bullish": 5,
    "naik tajam": 5, "naik kuat": 4, "gerak naik": 4, "potensi naik": 4,
    "layak beli": 4, "sangat bagus": 5, "bagus banget": 5,
    "cukup bagus": 3, "sangat baik": 5, "baik sekali": 5,
    "sangat menarik": 4, "sinyal positif": 5, "momentum positif": 5,
    "minat tinggi": 4, "adopsi tinggi": 4, "volume naik": 4,
    "lonjak": 4, "harga naik": 5, "target tercapai": 5, "profit besar": 5
}

custom_negative = {
    "jelek": 3, "buruk": 3, "parah": 3, "payah": 3, "lemah": 3,
    "mahal": 2, "lambat": 2, "lemot": 2, "gagal": 4, "rusak": 3,
    "masalah": 3, "risiko": 3, "bahaya": 4, "negatif": 3, "kecewa": 3,
    "mengecewakan": 4, "rugi": 4, "loss": 4, "turun": 3, "jatuh": 3,
    "anjlok": 5, "merosot": 4, "bearish": 4, "dump": 4, "crash": 5,
    "scam": 5, "fraud": 5, "penipuan": 5, "rugpull": 5, "panic": 3,
    "sell": 3, "panic sell": 5, "cut loss": 5, "downtrend": 4,
    "breakdown": 4, "support jebol": 5, "tren bearish": 5,
    "sinyal bearish": 5, "turun tajam": 5, "turun kuat": 4,
    "harga turun": 5, "potensi turun": 4, "tekan jual": 4,
    "tekanan jual": 5, "jual besar": 4, "volume turun": 4,
    "tidak bagus": 4, "tidak baik": 4, "sangat buruk": 5,
    "jelek banget": 5, "sangat jelek": 5
}

positive_dict.update(custom_positive)
negative_dict.update(custom_negative)

# =========================================================
# 6. NEGASI
# =========================================================
negasi_words = {
    "tidak", "tak", "bukan", "ga", "gak", "nggak", "enggak", "tdk", "kurang"
}

# =========================================================
# 7. FUNGSI LABELING
#    - cek unigram
#    - cek bigram / frasa 2 kata
# =========================================================
def sentiment_scoring(text):
    words = str(text).split()

    pos_score = 0
    neg_score = 0
    matched_pos = 0
    matched_neg = 0

    # cek frasa 2 kata
    for i in range(len(words) - 1):
        bigram = words[i] + " " + words[i + 1]

        if bigram in positive_dict:
            pos_score += positive_dict[bigram]
            matched_pos += 1

        if bigram in negative_dict:
            neg_score += negative_dict[bigram]
            matched_neg += 1

    # cek kata tunggal
    for i, word in enumerate(words):
        prev_word = words[i - 1] if i > 0 else ""

        if word in positive_dict:
            weight = positive_dict[word]

            if prev_word in negasi_words:
                neg_score += weight
                matched_neg += 1
            else:
                pos_score += weight
                matched_pos += 1

        elif word in negative_dict:
            weight = negative_dict[word]

            if prev_word in negasi_words:
                pos_score += weight
                matched_pos += 1
            else:
                neg_score += weight
                matched_neg += 1

    final_score = pos_score - neg_score

    # aturan label
    if matched_pos == 0 and matched_neg == 0:
        label = "netral"
    elif final_score >= 2:
        label = "positif"
    elif final_score <= -2:
        label = "negatif"
    else:
        label = "netral"

    return pd.Series([pos_score, neg_score, final_score, label])

# =========================================================
# 8. PROSES LABELING
# =========================================================
df[["pos_score", "neg_score", "final_score", "label"]] = df["hasil_preprocessing"].apply(sentiment_scoring)

# =========================================================
# 9. AMBIL HANYA KOLOM PENTING
# =========================================================
hasil = df[["hasil_preprocessing", "pos_score", "neg_score", "final_score", "label"]].copy()

print("\nDistribusi label:")
print(hasil["label"].value_counts())

print("\nContoh hasil:")
print(hasil.head(10))

# =========================================================
# 10. SIMPAN CSV
# =========================================================
hasil.to_csv("hasil_labeling_otomatis_bersih.csv", sep=";", index=False, encoding="utf-8-sig")

# =========================================================
# 11. SIMPAN FORMAT EXCEL
# =========================================================
hasil.to_excel("hasil_labeling_otomatis_bersih.xlsx", index=False)

print("\nFile berhasil disimpan:")
print("- hasil_labeling_otomatis_bersih.csv")
print("- hasil_labeling_otomatis_bersih.xlsx")

Jumlah data: 1127

Distribusi label:
label
positif    740
netral     216
negatif    171
Name: count, dtype: int64

Contoh hasil:
                                 hasil_preprocessing  pos_score  neg_score  \
0  tanggal januari total pasok toncoin ton mata u...          4          9   
1  toncoin hadap tekan jual potensi sinyal balik ...          7          7   
2  toncoin ton milik sinyal tarik perhati khusus ...         33          3   
3    bagus mes toncoin nad toncoin chog pesan domain          5          0   
4  bagus mes toncoin mon toncoin moyaki pesan domain          5          0   
5     tren bearish toncoin ton lanjut prediksi harga          8         11   
6   toncoin k ton pindah bursa harus dagang khawatir          2          5   
7   toncoin k ton pindah bursa harus dagang khawatir          2          5   
8                tren turun harga toncoin ton lanjut          6          2   
9                tren turun harga toncoin ton lanjut          6          2   

   final_sco